In [1]:
#!/usr/bin/env python3
"""Archive the 72 resolved off-chain metadata docs -> metadata_raw_snapshot.jsonl (Colab)."""
import os, json, base64, hashlib, datetime, urllib.parse
import pandas as pd, requests

DATA_DIR = "/content"                                  # upload agent_metadata.csv here
OUT      = "/content/metadata_raw_snapshot.jsonl"
TIMEOUT  = 20

md = pd.read_csv(f"{DATA_DIR}/agent_metadata.csv")
now = datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

records, ok, fail = [], 0, 0
for _, r in md.iterrows():
    aid = int(r["agent_id"]); url = str(r["metadata_url_resolved"])
    rec = {"agent_id": aid, "resolved_url": url, "fetched_at": now}
    try:
        if url.startswith("data:"):
            header, _, data = url.partition(",")
            raw = base64.b64decode(data) if ";base64" in header else urllib.parse.unquote(data).encode("utf-8")
            rec["source_type"] = "data_uri_onchain"; rec["http_status"] = None
        else:
            resp = requests.get(url, timeout=TIMEOUT, headers={"User-Agent": "erc8004-archive/1.0"})
            resp.raise_for_status()
            raw = resp.content
            rec["http_status"] = resp.status_code
            rec["source_type"] = "ipfs" if "/ipfs/" in url else "http"
        rec["sha256"] = hashlib.sha256(raw).hexdigest()
        rec["content_bytes"] = len(raw)
        try:
            rec["raw_text"] = raw.decode("utf-8")
        except UnicodeDecodeError:
            rec["raw_base64"] = base64.b64encode(raw).decode("ascii")
        if "/ipfs/" in url:
            rec["ipfs_cid"] = url.split("/ipfs/")[1].split("/")[0].split("?")[0]
        rec["status"] = "ok"; ok += 1
    except Exception as e:
        rec["status"] = "failed"; rec["error"] = repr(e); fail += 1
    records.append(rec)

with open(OUT, "w", encoding="utf-8") as f:
    for rec in records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

by_type = {}
for rec in records:
    if rec["status"] == "ok":
        by_type[rec["source_type"]] = by_type.get(rec["source_type"], 0) + 1
print(f"wrote {OUT}")
print(f"archived OK: {ok} / {len(md)}  (by source: {by_type})  | unreachable: {fail}")
if fail:
    print("unreachable agents:", [r['agent_id'] for r in records if r['status'] == 'failed'])

# download to your computer
from google.colab import files
files.download(OUT)

wrote /content/metadata_raw_snapshot.jsonl
archived OK: 69 / 72  (by source: {'http': 57, 'ipfs': 9, 'data_uri_onchain': 3})  | unreachable: 3
unreachable agents: [6817, 6815, 6887]


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>